In [ ]:
from thbsplines.hierarchical_space import HierarchicalSpace
import numpy as np
import scipy.sparse as sp
import dolfinx
from mpi4py import MPI
import basix.ufl
import pyvista

import numba
from dolfinx.jit import ffcx_jit
from dolfinx import default_real_type, default_scalar_type
rtype = default_real_type
dtype = default_scalar_type
import ufl

import numpy.typing as npt


from thbsplines.refinement import refine
from thbsplines.fenicsx.mesh import build_mesh, FastMidpointMapper
from thbsplines.fenicsx.functionspace import build_dofmap, fill_function_space, create_spline_space
from thbsplines.fenicsx.solvers import solve_problem
from thbsplines.fenicsx.adaptivity import dorfler_marking
from thbsplines.fenicsx.kernels import make_linear_kernel, make_bilinear_kernel

In [ ]:
n_refinements = 1
p0 = 2
knots1 = np.array([0,0,0, 0.5,0.5,1, 1, 1], dtype=np.float64)
#knots1 = np.array([-1, -1, -1, 0,0,1,1,1], dtype=np.float64)
knots1 = refine(knots1, p=p0, n_times=n_refinements)
log_initial_mesh_size = np.log2(np.max(np.diff(knots1)))
knots2 = np.array([0,0,0,0.5,1,1,1], dtype=np.float64)
#knots2 = np.array([-1, -1, -1, 0,0,1,1,1], dtype=np.float64)
knots2 = refine(knots2, p0, n_times=n_refinements-1)
err_cells = {}
hs = HierarchicalSpace(knots=[knots1, knots2], degrees=[p0])


In [ ]:
for level, cells in err_cells.items():
    #print(cells)
    hs.refine(cells, level, refine_neighbours=False, refine_T_neighbours=True, m=3)
hs.hmesh.plot_cells()

In [ ]:
def map_uv_to_xy_small(uv_points, nodes_per_cell=4):
    original_shape = uv_points.shape
    uv_flat = uv_points.reshape(-1, 2)
    
    u = uv_flat[:, 0]
    v = uv_flat[:, 1]
    xy_flat = np.zeros_like(uv_flat)

    # ---------------------------------------------------------
    # MACRO PATCH 1: Left Half (u < 0.5)
    # ---------------------------------------------------------
    left_mask = u < 0.5
    u_L = u[left_mask]
    v_L = v[left_mask]
    # P(U,V) = (1-U)(1-V)P00 + U(1-V)P10 + (1-U)VP01 + UVP11
    # for this specific problem, 
    # P00=(0,-1), 
    # P10 =(0,0)
    # P01 = (-1, -1), 
    # P11 = (-1, 1)
    
    xy_flat[left_mask, 0] = -v_L
    # substituting U=2u, V=v
    xy_flat[left_mask, 1] = 2 * u_L * v_L + 2 * u_L - 1

    # ---------------------------------------------------------
    # MACRO PATCH 2: Right Half (u >= 0.5)
    # ---------------------------------------------------------
    right_mask = u >= 0.5
    u_R = u[right_mask]
    v_R = v[right_mask]
    
    xy_flat[right_mask, 0] = 2 * u_R * v_R + 2 * u_R - 2 * v_R - 1
    xy_flat[right_mask, 1] = v_R
    
    return xy_flat.reshape(original_shape)


disconnected_mesh, thb_operators, N_max, physical_cells_midpoints = build_mesh(hs=hs, mapping=map_uv_to_xy_small)

In [ ]:
# topology, cell_types, geometry = dolfinx.plot.vtk_mesh(disconnected_mesh)
# grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# plotter = pyvista.Plotter()
# plotter.add_mesh(grid.shrink(.95), show_edges=False, color="#03bb85")
# plotter.view_xy()
# plotter.show(jupyter_backend="static")
# print(f"Number of points in PyVista grid: {grid.n_points}")

In [ ]:
def outer_boundary(x):
    on_left = np.isclose(x[0], -1.)
    on_bottom = np.isclose(x[1], -1.)
    on_right = np.isclose(x[0], 1.) & (x[1]>=-1e-10)
    on_top = np.isclose(x[1], 1.) & (x[0]<=1.)
    return on_left|on_bottom|on_right|on_top

legendre_elt = basix.ufl.element(
    "DG",
    "quadrilateral",
    degree=p0,
    lagrange_variant=basix.LagrangeVariant.legendre
)
V = dolfinx.fem.functionspace(disconnected_mesh, legendre_elt)
print(f"Number of degrees of freedom: {V.dofmap.index_map.size_global}")
facet_dim = disconnected_mesh.topology.dim-1
boundary_facets = dolfinx.mesh.locate_entities_boundary(disconnected_mesh, facet_dim, outer_boundary)
custom_metadata = {"quadrature_degree": 6}
#ds = ufl.Measure("ds", domain=disconnected_mesh, subdomain_data=([1, boundary_facets]), metadata=custom_metadata)
dx_custom = ufl.Measure("dx", domain=disconnected_mesh, metadata=custom_metadata)


u,v = ufl.TrialFunction(V), ufl.TestFunction(V) 
my_x = ufl.SpatialCoordinate(disconnected_mesh)
# f = dolfinx.fem.Function(V)
#f.interpolate(lambda x: x[0]*x[1]+0.9*x[0]**2-.7)
f =  1./(1.*ufl.exp((my_x[0]+.0625)**2 + (my_x[1]-.0625)**2))
a0 = ufl.inner(u,v)*dx_custom
f0 = ufl.inner(f,v)*dx_custom
f_square_integral = dolfinx.fem.assemble_scalar(dolfinx.fem.form(ufl.inner(f, f)*dx_custom))
f_sq_integral = np.sqrt(disconnected_mesh.comm.allreduce(f_square_integral, op=MPI.SUM))


msh = disconnected_mesh
ufcxa0, _, _ = ffcx_jit(msh.comm, a0, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernela0 = getattr(ufcxa0.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore
ufcxf0, _, _ = ffcx_jit(msh.comm, f0, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernelf0 = getattr(ufcxf0.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore


In [ ]:
dofmap, padded_cells_to_dofs = build_dofmap(hierarchical_space=hs, mesh=disconnected_mesh, 
                                            N_max=N_max, morton=False)
custom_mapping = FastMidpointMapper(hs, physical_cells_midpoints)
C_func, C_space = fill_function_space(hierachical_space=hs, mesh=disconnected_mesh,
                                      N_max=N_max, thb_operators=thb_operators, mapping_function=custom_mapping)
V_spline = create_spline_space(cells_to_dofs=padded_cells_to_dofs, mesh=disconnected_mesh,
                               N_max=N_max, mult_factor=1)

local_dofs = (hs.degrees[0]+1)**2
tabulate_A = make_bilinear_kernel(dtype, rtype, ufcx_kernel=kernela0, padded_dofs=N_max, local_dofs=local_dofs)
tabulate_L = make_linear_kernel(dtype, rtype, ufcx_kernel=kernelf0, padded_dofs=N_max, local_dofs=local_dofs)

In [ ]:
facet_dim = msh.topology.dim-1
boundary_facets = dolfinx.mesh.locate_entities_boundary(msh, facet_dim, outer_boundary)
msh.topology.create_connectivity(facet_dim, msh.topology.dim)
msh.topology.create_connectivity(msh.topology.dim, facet_dim)

# dictionary of facet->cell
f_to_c = msh.topology.connectivity(facet_dim, msh.topology.dim)
# dictionary of cell->array[facets]
c_to_f = msh.topology.connectivity(msh.topology.dim, facet_dim)

boundary_entities = []
# Loop over all edges that belong to our exterior
for ff in boundary_facets:
    # returns the cells linked to this edge
    cells = f_to_c.links(ff)
    # Exterior facets only have 1 attached cell, 
    # hence get the first one 
    c = cells[0] 
    
    # Find the local index (e.g., 0, 1, 2, or 3 for quads) of facet f within cell c
    local_f = np.where(c_to_f.links(c) == ff)[0][0]
    #print(f"f={f}, cell={c}, local_f = {local_f}")
    
    boundary_entities.extend([c, local_f])

# FEniCSx custom form arrays must be typed as int32
boundary_entities = np.array(boundary_entities, dtype=np.int32)

In [ ]:
formtype = dolfinx.fem.form_cpp_class(dtype)  # type: ignore
# Gets the number of cells for which each individual core is responsible for.
cells = np.arange(msh.topology.index_map(msh.topology.dim).size_local, dtype=np.int32)

# The 4th argument np.array([...], dtype=np.int8) is the 
# active coefficients array. It lists which indices from the 
# coefficients list should be packed into the w_ pointer that the kernel receives.
integrals = {dolfinx.fem.IntegralType.cell: [
    (0, tabulate_A.address, cells, np.array([0], dtype=np.int8))]}

a_cond = dolfinx.fem.Form( # We are not forming anything yet, this is a recipe
    formtype( # selectes the correct floating-point precision
        spaces=[V_spline._cpp_object, 
                V_spline._cpp_object]
            , # trial and test spaces, determines the size of A_
        integrals=integrals, #this is a dictionary, and we are passing the adress of tabulate_A() here
        coefficients=[C_func._cpp_object
                    ], # weights w_, holds C@T
              constants=[],
              need_permutation_data=False,
              entity_maps=[], 
              mesh=msh._cpp_object)
)

integrals_rhs = {dolfinx.fem.IntegralType.cell: [(0, tabulate_L.address, cells, np.array([0], dtype=np.int8))]}
l_cond = dolfinx.fem.Form(
    formtype(
        spaces=[V_spline._cpp_object], # test space, determines the size of b_
        integrals=integrals_rhs, #give the adress of tabulate_b
        coefficients=[C_func._cpp_object], # holds the evaluations of f at the correct points, as well as C@T
        constants=[], need_permutation_data=False, entity_maps=[], mesh=msh._cpp_object
    )
)

In [ ]:
def get_spline_indices_on_inner_corner_turn(hs, dofmap):
    dirichlet_indices = {}
    for level in range(hs.nlevels):
        hs.level_spaces[level].construct_basis()
        basis = hs.level_spaces[level].basis
        current_dirichlet_indices = np.isclose(basis[:, 0, :-1], np.zeros((hs.degrees[0]+1)))
        current_dirichlet_indices = np.all(current_dirichlet_indices, axis=-1)
        current_dirichlet_indices = np.nonzero(current_dirichlet_indices)[0]
        dirichlet_indices[level] = np.intersect1d(hs.truly_active[level],
                                                  current_dirichlet_indices,
                                                  assume_unique=True)
    pass

    forbidden_indices = np.array([dofmap[level,idx] for level in dirichlet_indices 
              for idx in dirichlet_indices[level]],
                        dtype=np.int32)
    return forbidden_indices

forbidden_indices = get_spline_indices_on_inner_corner_turn(hs, dofmap)

In [ ]:
x_vec, A = solve_problem(hs=hs, a=a_cond, rhs=l_cond, dirichlet_indices=forbidden_indices,
                         dummy_index=np.max(padded_cells_to_dofs), V_spline=V_spline,
                         iterative=False, return_A=True)

In [ ]:
u_dg = dolfinx.fem.Function(V)
c_values = C_func.x.array.reshape((-1, N_max, (hs.degrees[0]+1)**2))

# Map the global B-spline coefficients back to local Legendre coefficients
for local_idx in range(msh.topology.index_map(msh.topology.dim).size_local):
    # Get global B-spline dof indices for this cell
    spline_dofs = padded_cells_to_dofs[local_idx]
    
    # Extract the B-spline coefficients for this cell
    u_spline_local = x_vec[spline_dofs]
    
    # Get the local transformation matrix G for this cell
    G = c_values[local_idx, :, :]
    
    # Transform B-spline to DG: mathematically, the kernel does A = G @ A0 @ G.T
    # This implies the coefficient mapping is u_dg = G.T @ u_spline
    u_dg_local = G.T @ u_spline_local
    
    # Assign to the standard DG function
    dg_dofs = V.dofmap.cell_dofs(local_idx)
    u_dg.x.array[dg_dofs] = u_dg_local

u_dg.x.scatter_forward()


# Compute exact L2 error using FEniCSx standard UFL
error_form = dolfinx.fem.form(ufl.inner(f - u_dg, f - u_dg) * dx_custom)
error_sq = dolfinx.fem.assemble_scalar(error_form)
exact_l2_error = np.sqrt(disconnected_mesh.comm.allreduce(error_sq, op=MPI.SUM))

print(f"Exact L2 Error (via DG projection): {exact_l2_error:.2e}")
print(f"Relative error = {exact_l2_error/f_sq_integral:.2e}")

In [ ]:
# Create a DG0 space (one value per cell)
V_error = dolfinx.fem.functionspace(disconnected_mesh, ("DG", 0))
v = ufl.TestFunction(V_error)
hQ = ufl.CellDiameter(disconnected_mesh)
volume_form = dolfinx.fem.form(1.0*v*dx_custom)
cell_volumes = dolfinx.fem.assemble_vector(volume_form).array
#print(f"cell_volumes = {cell_volumes[:10]}")
# Define the local L2 error form: integral of (f - u_dg)^2 per cell
# Note: We multiply by the test function 'v' to pick out each cell's contribution
local_error_form = dolfinx.fem.form(ufl.inner(f - u_dg, f - u_dg) *v * dx_custom)
err_cells = dorfler_marking(hs, 0.3, local_error_form=local_error_form)

In [ ]:
# import dolfinx.plot
# import pyvista

# 1. Create a "Nodal" DG space of the same degree for plotting
# By default, DG with no variant specified uses Lagrange (nodal)
# v_plot_elt = basix.ufl.element(
#     "DG", 
#     "quadrilateral", 
#     degree=p0+2
# )
# V_plot = dolfinx.fem.functionspace(disconnected_mesh, v_plot_elt)

# 2. Interpolate your computed solution (u_dg) into the nodal space
#u_plot = dolfinx.fem.Function(V_plot)
# error_ufl = ufl.ln(ufl.sqrt((u_dg-f)**2)+1e-13)
#error_ufl = u_dg
# error_expr = dolfinx.fem.Expression(error_ufl, V_plot.element.interpolation_points)
# u_error = dolfinx.fem.Function(V_plot)
# u_error.interpolate(error_expr)

# 3. Now use V_plot for the VTK mesh generation
# topology, cell_types, geometry = dolfinx.plot.vtk_mesh(V_plot)
# grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# 4. Attach the interpolated values
# grid.point_data["u"] = u_error.x.array.real
# grid.set_active_scalars("u")

# 5. Plotting (with a 'shrink' to see your disconnected mesh boundaries!)
# plotter = pyvista.Plotter()
# grid_shrink = grid.shrink(0.95) # This makes the "disconnected" nature visible
# plotter.add_mesh(grid_shrink, show_edges=False, cmap="turbo")
# plotter.view_xy()
# plotter.show(jupyter_backend="static")
# plotter.show()

# from pyvista.trame.jupyter import launch_server
# pyvista.set_jupyter_backend('client')
# warped_grid = grid.warp_by_scalar("u", factor=0.5) 

# # If you still want to see the gaps between cells:
# grid_shrink = warped_grid.shrink(0.95)

# plotter.add_mesh(grid_shrink, show_edges=False, cmap="viridis", lighting=True)

# # Set a nice 3D camera angle instead of view_xy()
# plotter.camera_position = 'iso' 
# await launch_server().ready
# plotter.show()